In [0]:
dbutils.widgets.text("catalog", "claudecatalog", "Catálogo")
dbutils.widgets.text("schema", "supply_chain", "Schema")
dbutils.widgets.text("volume", "raw_files", "Volume")
dbutils.widgets.text("file_name", "DataCoSupplyChainDataset.csv", "Archivo CSV")

catalog = dbutils.widgets.get("catalog")
schema = dbutils.widgets.get("schema")
volume = dbutils.widgets.get("volume")
file_name = dbutils.widgets.get("file_name")

file_path = f"/Volumes/{catalog}/{schema}/{volume}/{file_name}"
print(f"Leyendo desde: {file_path}")

Leyendo desde: /Volumes/claudecatalog/supply_chain/raw_files/DataCoSupplyChainDataset.csv


Esto convierte, por ejemplo:

"Days for shipping (real)" → Days_for_shipping_real
"Benefit per order" → Benefit_per_order
"Customer Zipcode" → Customer_Zipcode

Importante: esto es una decisión de transformación, así que técnicamente ya no es "Bronze puro" en el sentido más estricto (Bronze idealmente es 1:1 con la fuente). Pero es una excepción aceptada en la industria: renombrar columnas por restricciones técnicas de almacenamiento (no por lógica de negocio) se hace en Bronze porque si no, ni siquiera puedes persistir los datos. Documenta esto en tu README como una decisión consciente — es exactamente el tipo de criterio que se evalúa en una entrevista técnica.

In [0]:
df_raw = (
    spark.read
    .option("header", "true")
    .option("inferSchema", "true")
    .option("encoding", "ISO-8859-1")
    .csv(file_path)
)

print(f"Filas leídas: {df_raw.count()}")
display(df_raw.limit(5))

Filas leídas: 180519


Type,Days for shipping (real),Days for shipment (scheduled),Benefit per order,Sales per customer,Delivery Status,Late_delivery_risk,Category Id,Category Name,Customer City,Customer Country,Customer Email,Customer Fname,Customer Id,Customer Lname,Customer Password,Customer Segment,Customer State,Customer Street,Customer Zipcode,Department Id,Department Name,Latitude,Longitude,Market,Order City,Order Country,Order Customer Id,order date (DateOrders),Order Id,Order Item Cardprod Id,Order Item Discount,Order Item Discount Rate,Order Item Id,Order Item Product Price,Order Item Profit Ratio,Order Item Quantity,Sales,Order Item Total,Order Profit Per Order,Order Region,Order State,Order Status,Order Zipcode,Product Card Id,Product Category Id,Product Description,Product Image,Product Name,Product Price,Product Status,shipping date (DateOrders),Shipping Mode
DEBIT,3,4,91.25,314.6400146,Advance shipping,0,73,Sporting Goods,Caguas,Puerto Rico,XXXXXXXXX,Cally,20755,Holloway,XXXXXXXXX,Consumer,PR,5365 Noble Nectar Island,725,2,Fitness,18.2514534,-66.03705597,Pacific Asia,Bekasi,Indonesia,20755,1/31/2018 22:56,77202,1360,13.10999966,0.039999999,180517,327.75,0.289999992,1,327.75,314.6400146,91.25,Southeast Asia,Java Occidental,COMPLETE,null,1360,73,null,http://images.acmesports.sports/Smart+watch,Smart watch,327.75,0,2/3/2018 22:56,Standard Class
TRANSFER,5,4,-249.0899963,311.3599854,Late delivery,1,73,Sporting Goods,Caguas,Puerto Rico,XXXXXXXXX,Irene,19492,Luna,XXXXXXXXX,Consumer,PR,2679 Rustic Loop,725,2,Fitness,18.27945137,-66.0370636,Pacific Asia,Bikaner,India,19492,1/13/2018 12:27,75939,1360,16.38999939,0.050000001,179254,327.75,-0.800000012,1,327.75,311.3599854,-249.0899963,South Asia,Rajastán,PENDING,null,1360,73,null,http://images.acmesports.sports/Smart+watch,Smart watch,327.75,0,1/18/2018 12:27,Standard Class
CASH,4,4,-247.7799988,309.7200012,Shipping on time,0,73,Sporting Goods,San Jose,EE. UU.,XXXXXXXXX,Gillian,19491,Maldonado,XXXXXXXXX,Consumer,CA,8510 Round Bear Gate,95125,2,Fitness,37.29223251,-121.881279,Pacific Asia,Bikaner,India,19491,1/13/2018 12:06,75938,1360,18.03000069,0.059999999,179253,327.75,-0.800000012,1,327.75,309.7200012,-247.7799988,South Asia,Rajastán,CLOSED,null,1360,73,null,http://images.acmesports.sports/Smart+watch,Smart watch,327.75,0,1/17/2018 12:06,Standard Class
DEBIT,3,4,22.86000061,304.8099976,Advance shipping,0,73,Sporting Goods,Los Angeles,EE. UU.,XXXXXXXXX,Tana,19490,Tate,XXXXXXXXX,Home Office,CA,3200 Amber Bend,90027,2,Fitness,34.12594605,-118.2910156,Pacific Asia,Townsville,Australia,19490,1/13/2018 11:45,75937,1360,22.94000053,0.07,179252,327.75,0.079999998,1,327.75,304.8099976,22.86000061,Oceania,Queensland,COMPLETE,null,1360,73,null,http://images.acmesports.sports/Smart+watch,Smart watch,327.75,0,1/16/2018 11:45,Standard Class
PAYMENT,2,4,134.2100067,298.25,Advance shipping,0,73,Sporting Goods,Caguas,Puerto Rico,XXXXXXXXX,Orli,19489,Hendricks,XXXXXXXXX,Corporate,PR,8671 Iron Anchor Corners,725,2,Fitness,18.25376892,-66.03704834,Pacific Asia,Townsville,Australia,19489,1/13/2018 11:24,75936,1360,29.5,0.090000004,179251,327.75,0.449999988,1,327.75,298.25,134.2100067,Oceania,Queensland,PENDING_PAYMENT,null,1360,73,null,http://images.acmesports.sports/Smart+watch,Smart watch,327.75,0,1/15/2018 11:24,Standard Class


In [0]:
import re

def sanitize_column_name(col_name):
    # Reemplaza espacios y caracteres inválidos por guion bajo
    clean = re.sub(r'[ ,;{}()\n\t=]+', '_', col_name)
    # Evita guiones bajos duplicados y al final
    clean = re.sub(r'_+', '_', clean).strip('_')
    return clean

new_column_names = [sanitize_column_name(c) for c in df_raw.columns]
df_raw = df_raw.toDF(*new_column_names)

display(df_raw.columns)

_1
Type
Days_for_shipping_real
Days_for_shipment_scheduled
Benefit_per_order
Sales_per_customer
Delivery_Status
Late_delivery_risk
Category_Id
Category_Name
Customer_City


Celda 3 — Escribir la tabla Bronze en Delta

Por qué: aquí agregamos las columnas de auditoría que mencionamos en el roadmap (_ingested_at, _source_file) — esto es estándar en cualquier capa Bronze real, porque te permite rastrear de dónde vino cada fila y cuándo se cargó, algo esencial para debugging y linaje de datos.

In [0]:
from pyspark.sql.functions import current_timestamp, lit

df_bronze = (
    df_raw
    .withColumn("_ingested_at", current_timestamp())
    .withColumn("_source_file", lit(file_name))
)

bronze_table = f"{catalog}.{schema}.bronze_orders"

(
    df_bronze.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(bronze_table)
)

print(f"Tabla creada: {bronze_table}")

Tabla creada: claudecatalog.supply_chain.bronze_orders


In [0]:
display(spark.table(bronze_table))

Type,Days_for_shipping_real,Days_for_shipment_scheduled,Benefit_per_order,Sales_per_customer,Delivery_Status,Late_delivery_risk,Category_Id,Category_Name,Customer_City,Customer_Country,Customer_Email,Customer_Fname,Customer_Id,Customer_Lname,Customer_Password,Customer_Segment,Customer_State,Customer_Street,Customer_Zipcode,Department_Id,Department_Name,Latitude,Longitude,Market,Order_City,Order_Country,Order_Customer_Id,order_date_DateOrders,Order_Id,Order_Item_Cardprod_Id,Order_Item_Discount,Order_Item_Discount_Rate,Order_Item_Id,Order_Item_Product_Price,Order_Item_Profit_Ratio,Order_Item_Quantity,Sales,Order_Item_Total,Order_Profit_Per_Order,Order_Region,Order_State,Order_Status,Order_Zipcode,Product_Card_Id,Product_Category_Id,Product_Description,Product_Image,Product_Name,Product_Price,Product_Status,shipping_date_DateOrders,Shipping_Mode,_ingested_at,_source_file
PAYMENT,2,4,20.70000076,44.99000168,Advance shipping,0,17,Cleats,Caguas,Puerto Rico,XXXXXXXXX,Cynthia,5913,Daniel,XXXXXXXXX,Consumer,PR,1658 Rustic Elk Lane,725,4,Apparel,18.23004913,-66.3705368,Europe,Milan,Italia,5913,6/30/2017 9:45,62436,365,15.0,0.25,156057,59.99000168,0.460000008,1,59.99000168,44.99000168,20.70000076,Southern Europe,Lombardía,PENDING_PAYMENT,null,365,17,null,http://images.acmesports.sports/Perfect+Fitness+Perfect+Rip+Deck,Perfect Fitness Perfect Rip Deck,59.99000168,0,7/2/2017 9:45,Standard Class,2026-08-24T17:58:18.650Z,DataCoSupplyChainDataset.csv
PAYMENT,5,4,-77.98999786,97.48999786,Late delivery,1,18,Men's Footwear,Caguas,Puerto Rico,XXXXXXXXX,Olivia,6720,Moran,XXXXXXXXX,Consumer,PR,7851 Stony Prairie Highway,725,4,Apparel,18.28904724,-66.37052155,Europe,Ponteareas,España,6720,8/4/2017 4:37,64819,403,32.5,0.25,162020,129.9900055,-0.800000012,1,129.9900055,97.48999786,-77.98999786,Southern Europe,Galicia,PENDING_PAYMENT,null,403,18,null,http://images.acmesports.sports/Nike+Men%27s+CJ+Elite+2+TD+Football+Cleat,Nike Men's CJ Elite 2 TD Football Cleat,129.9900055,0,8/9/2017 4:37,Standard Class,2026-08-24T17:58:18.650Z,DataCoSupplyChainDataset.csv
PAYMENT,6,4,80.34999847,267.8299866,Late delivery,1,63,Children's Clothing,Caguas,Puerto Rico,XXXXXXXXX,Shafira,13988,Battle,XXXXXXXXX,Consumer,PR,9688 Silent Lagoon Meadow,725,4,Apparel,18.24829483,-66.03704834,Europe,Alfortville,Francia,13988,10/25/2017 4:10,70435,1350,89.27999878,0.25,173750,357.1000061,0.300000012,1,357.1000061,267.8299866,80.34999847,Western Europe,Isla de Francia,PENDING_PAYMENT,null,1350,63,null,http://images.acmesports.sports/Children+heaters,Children's heaters,357.1000061,0,10/31/2017 4:10,Standard Class,2026-08-24T17:58:18.650Z,DataCoSupplyChainDataset.csv
PAYMENT,4,4,3.369999886,44.99000168,Shipping on time,0,17,Cleats,Caguas,Puerto Rico,XXXXXXXXX,Joan,1243,Rosales,XXXXXXXXX,Consumer,PR,1777 High Alley,725,4,Apparel,18.21440888,-66.37060547,Europe,Bremen,Alemania,1243,8/10/2017 21:26,65278,365,15.0,0.25,163142,59.99000168,0.079999998,1,59.99000168,44.99000168,3.369999886,Western Europe,Bremen,PENDING_PAYMENT,null,365,17,null,http://images.acmesports.sports/Perfect+Fitness+Perfect+Rip+Deck,Perfect Fitness Perfect Rip Deck,59.99000168,0,8/14/2017 21:26,Standard Class,2026-08-24T17:58:18.650Z,DataCoSupplyChainDataset.csv
PAYMENT,3,4,-37.47999954,44.99000168,Advance shipping,0,17,Cleats,Caguas,Puerto Rico,XXXXXXXXX,Mary,5284,Smith,XXXXXXXXX,Consumer,PR,2876 Lazy Spring Corners,725,4,Apparel,18.25136757,-66.3706131,Europe,Nuremberg,Alemania,5284,6/19/2017 4:20,61667,365,15.0,0.25,154173,59.99000168,-0.829999983,1,59.99000168,44.99000168,-37.47999954,Western Europe,Bavaria,PENDING_PAYMENT,null,365,17,null,http://images.acmesports.sports/Perfect+Fitness+Perfect+Rip+Deck,Perfect Fitness Perfect Rip Deck,59.99000168,0,6/22/2017 4:20,Standard Class,2026-08-24T17:58:18.650Z,DataCoSupplyChainDataset.csv
PAYMENT,4,4,5.079999924,44.99000168,Shipping on time,0,17,Cleats,Caguas,Puerto Rico,XXXXXXXXX,Hannah,809,Avila,XXXXXXXXX,Consumer,PR,3112 Easy Turnabout,725,4,Apparel,18.22711754,-66.

In [0]:
%sql
use claudecatalog.supply_chain;
select count(*)
from bronze_orders

count(*)
180519
